# Classificação de operações fraudulentas
O banco Global Trust coletou 284,807 transações, mas apenas 492 foram identificadas como fraudulentas (0.172%).

* A base já passou por vetorização com PCA;
* A coluna "time" representa o tempo em segundos entre as transações;
* A coluna "amount" representa o valor em moeda da transação;
* Na variável alvo, 0 são transações legítimas e 1 são operações fraudulentas

Paleta de cores:
* Cinza médio: #6B7280
* Cinza claro: #9CA3AF
* Preto: #1F2937
* Branco: #F3F4F6

In [24]:
# Biblioteca de manipulação de dados
import pandas as pd
import numpy as np
from scipy.stats import loguniform, uniform, randint

# Biblioteca de visualização de dados
import plotly.express as px
import plotly.graph_objects as go

# Biblioteca de preparação de dados
from imblearn.over_sampling import SMOTE
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler

# Biblioteca de modelos
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import LinearSVC

# Biblioteca de avaliação
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve, classification_report, confusion_matrix
import pickle

In [2]:
df = pd.read_csv('Base_M43_Pratique_CREDIT_CARD_FRAUD.csv')
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [3]:
am_cl = df.groupby('Amount')['Class'].value_counts().reset_index()
am_cl['Class'] = am_cl['Class'].map({0:'Legítimo', 1:'Fraude'})

fig = px.histogram(am_cl,
                   x='Class',
                   y='Amount',
                   color='Class',
                   color_discrete_map={'Legítimo':'#1F2937', 'Fraude':'#9CA3AF'})
fig.update_layout(
    template='seaborn',
    title='Classes x Quantia movimentada',
)

fig.show()

Esse gráfico indica que a maioria das transações são legítimas, indicando não ter correlação com quantia movimentada necessariamente

In [4]:
# Quantidade de registros de cada classe
df['Class'].value_counts()

Class
0    284315
1       492
Name: count, dtype: int64

A quantia de movimentações fraudulentas é ínfima em comparação com as legítimas

In [5]:
'''
Criando gráfico para descobrir se há correlação entre as operações fraudulentas e a quantia movimentada
'''
fraude = df[df['Class'] == 1]


fr_am = fraude.groupby('Class')['Amount'].value_counts().reset_index()


fig = px.histogram(fr_am,
                   x='Amount',
                   y='Class',
                   color='Class',
                   color_discrete_map={1:'#9CA3AF'})
fig.update_layout(
    template='seaborn',
    title='Fraudes x Quantia movimentada',
    xaxis_title='Quantia',
    yaxis_title='Contagem'
)
fig.show()

O Gráfico acima indica que a maioria das transações fraudulentas se encontram num espectro de pequeno valor (até 849 mais ou menos)

In [6]:
'''
É preciso saber quanto das pequenas movimentações são fraudes
'''
pqMov = df[df['Amount']<= 849]

pq_fr = pqMov.groupby(['Amount', 'Class']).size().reset_index(name='Count')

fig = px.histogram(pq_fr,
                   x='Amount',
                   y='Count',
                   color='Class',
                   color_discrete_map={1:'#9CA3AF', 0:'#1F2937'})
fig.update_layout(
    template='seaborn',
    title='Fraudes x  Pequena quantia',
    xaxis_title='Quantia',
    yaxis_title='Contagem',
    barmode='group'
)
fig.show()

In [7]:
pqMov['Class'].value_counts()

Class
0    280315
1       481
Name: count, dtype: int64

A quantia de pequenas transações legítimas é mais alta que as fraudulentas

In [8]:
# Gráficos de correlação de variáveis
corr_total = df.corr()
color_scale = [[0,'#1F2937'],[1,'#F3F4F6']]

fig = px.imshow(
    corr_total,
    text_auto=True,
    aspect='auto',
    color_continuous_scale=color_scale
)
fig.update_layout(
    title='Correlação de variáveis',
    template='seaborn'
)
fig.show()

Como a maioria das variáveis já foi vetorizada, a correlação com a variável target parece ser homogênea

## Treinamento e avaliação do modelo

In [9]:
'''
Como a variável target se encontra desbalançeada, antes de fazer o pipeline e ajuste de hiperparâmetros para treino, é necessário balancear o treino.
'''
x = df.drop(columns=['Class'])
y = df['Class']

x_train, x_test, y_train, y_test = train_test_split(x, y,
                                                    test_size=0.3,
                                                    train_size=0.15,
                                                    stratify=y,
                                                    random_state=42)

# Oversplampling o treino
smote = SMOTE(random_state=42)
x_train_resampled, y_train_resampled = smote.fit_resample(x_train, y_train)

In [10]:
x_train_resampled.info()

<class 'pandas.DataFrame'>
RangeIndex: 85294 entries, 0 to 85293
Data columns (total 30 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Time    85294 non-null  float64
 1   V1      85294 non-null  float64
 2   V2      85294 non-null  float64
 3   V3      85294 non-null  float64
 4   V4      85294 non-null  float64
 5   V5      85294 non-null  float64
 6   V6      85294 non-null  float64
 7   V7      85294 non-null  float64
 8   V8      85294 non-null  float64
 9   V9      85294 non-null  float64
 10  V10     85294 non-null  float64
 11  V11     85294 non-null  float64
 12  V12     85294 non-null  float64
 13  V13     85294 non-null  float64
 14  V14     85294 non-null  float64
 15  V15     85294 non-null  float64
 16  V16     85294 non-null  float64
 17  V17     85294 non-null  float64
 18  V18     85294 non-null  float64
 19  V19     85294 non-null  float64
 20  V20     85294 non-null  float64
 21  V21     85294 non-null  float64
 22  V22     8

In [11]:
y_train_resampled.value_counts()

Class
0    42647
1    42647
Name: count, dtype: int64

In [12]:
'''
Com o balançeamento feito, agora o pipeline e ajuste de hiperparâmetros será feito. Serão treinados os modelos de regressão logística, SVM, Random Forest e XGBoost
'''
# Regressão logística
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(random_state=42)),
])
param_lr = [{
    'lr__C': [0.001,0.01,0.1,1,10,100],
    'lr__solver': ['lbfgs'],
    'lr__max_iter':[100, 250, 500]
},
   {
    'lr__C': [0.001,0.01,0.1,1,10,100],
    'lr__penalty': ['elasticnet'],
    'lr__l1_ratio': [0.0, 0.5, 1.0],
    'lr__solver': ['saga'],
    'lr__max_iter':[100, 250, 500]
}]
search_lr = RandomizedSearchCV(
    estimator=pipe_lr,
    param_distributions=param_lr,
    n_iter=15,
    scoring='accuracy',
    cv=5,
    n_jobs=-1,
    random_state=42,
    refit=True
)
search_lr.fit(x_train_resampled, y_train_resampled)
best_lr = search_lr.best_estimator_

# Scores dos modelos em cross validation
scores_lr = cross_val_score(
    best_lr,
    x_train_resampled,
    y_train_resampled,
    cv=5,
    n_jobs=-1,
    scoring='recall'
)

pickle.dumps(best_lr)
print('Regressão logística (recall):', scores_lr.mean())

Regressão logística (recall): 0.9800923268891447


In [13]:
# SVM
pipe_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', LinearSVC( random_state=42)),
])
param_svm = {
    'svm__C': loguniform(1e-2, 1e2),
    'svm__loss': ['hinge', 'squared_hinge'],
    'svm__tol': loguniform(1e-5, 1e-3)
}
search_svm = RandomizedSearchCV(
    estimator=pipe_svm,
    param_distributions=param_svm,
    n_iter=15,
    scoring='accuracy',
    cv=5,
    n_jobs=-1,
    random_state=42,
    refit=True
)
search_svm.fit(x_train_resampled, y_train_resampled)
best_svm = search_svm.best_estimator_
scores_svm = cross_val_score(
    best_svm,
    x_train_resampled,
    y_train_resampled,
    cv=5,
    n_jobs=-1,
    scoring='recall'
)
print('SVM (recall):', scores_svm.mean())

D:\PyCharm 2025.3\EBAC\.venv\Lib\site-packages\sklearn\svm\_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


SVM (recall): 0.9808661133651043


In [14]:
# Random Forest
pipe_rf = Pipeline([
    ('rf', RandomForestClassifier(random_state=42)),
])
param_rf = {
    'rf__n_estimators': randint(50, 200),
    'rf__max_depth': [3, 10, 20, 30, 40, 50],
    'rf__min_samples_split': [2, 4, 8, 10, 12, 16, 18, 20],
    'rf__min_samples_leaf': [1,2,4,5,7,8,10],
    'rf__criterion': ['gini', 'entropy'],
    'rf__max_features': ['sqrt', 'log2'],
}
search_rf = RandomizedSearchCV(
    estimator=pipe_rf,
    param_distributions=param_rf,
    n_iter=15,
    scoring='accuracy',
    cv=5,
    n_jobs=-1,
    random_state=42,
    refit=True
)
search_rf.fit(x_train_resampled, y_train_resampled)
best_rf = search_rf.best_estimator_
scores_rf = cross_val_score(
    best_rf,
    x_train_resampled,
    y_train_resampled,
    cv=5,
    n_jobs=-1,
    scoring='recall'
)
print('Random Forest (recall):', scores_rf.mean())

Random Forest (recall): 1.0


In [15]:
# XGBoost
pipe_xgb = Pipeline([
    ('xgb', XGBClassifier(random_state=42)),
])
param_xgb = {
    'xgb__n_estimators': randint(100, 1000),
    'xgb__max_depth': randint(3, 15),
    'xgb__learning_rate': uniform(0.01, 0.29),
    'xgb__subsample': uniform(0.5, 0.5),
    'xgb__colsample_bytree': uniform(0.5, 0.5),
    'xgb__gamma': uniform(0, 10),
    'xgb__reg_alpha': uniform(0, 10),
    'xgb__reg_lambda': uniform(0, 10)
}
search_xgb = RandomizedSearchCV(
    estimator=pipe_xgb,
    param_distributions=param_xgb,
    n_iter=15,
    scoring='accuracy',
    cv=5,
    n_jobs=-1,
    random_state=42,
    refit=True
)
search_xgb.fit(x_train_resampled, y_train_resampled)
best_xgb = search_xgb.best_estimator_
scores_xgb = cross_val_score(
    best_xgb,
    x_train_resampled,
    y_train_resampled,
    cv=5,
    n_jobs=-1,
    scoring='recall'
)
print('XGBoost (recall):', scores_xgb.mean())

XGBoost (recall): 1.0


O treinamento demonstrou que todos os modelos se saíram bem em identificar os verdadeiros positivos. Isso é um bom indicativo de modelo que pode identificar fraudes.

# Teste e escolha do modelo a ser usado pelo banco

In [20]:
# Teste do modelo de regressão logística
y_pred_lr = best_lr.predict(x_test)

# Avaliação do modelo com o teste
report_lr = classification_report(y_test, y_pred_lr)
print('Relatório de classificação da Regressão Logística')
print(report_lr)

# Matriz de confusão
conf_lr = confusion_matrix(y_test, y_pred_lr)
classes = ['Legítimo', 'Fraude']

fig = px.imshow(
    conf_lr,
    text_auto=True,
    aspect='auto',
    template='seaborn',
    x=classes,
    y=classes,
    labels= dict(x='Predição', y='Real', color='Contagem'),
    color_continuous_scale= color_scale,
    title='Matriz de confusão - Regressão Logística'

)
fig.show()

Relatório de classificação da Regressão Logística
              precision    recall  f1-score   support

           0       1.00      0.99      1.00     85295
           1       0.14      0.88      0.24       148

    accuracy                           0.99     85443
   macro avg       0.57      0.93      0.62     85443
weighted avg       1.00      0.99      0.99     85443



In [21]:
# Teste do modelo de SVM
y_pred_svm = best_svm.predict(x_test)

# Avaliação do modelo com o teste
report_svm = classification_report(y_test, y_pred_svm)
print('Relatório de classificação do SVM')
print(report_svm)

# Matriz de confusão
conf_svm = confusion_matrix(y_test, y_pred_svm)
classes = ['Legítimo', 'Fraude']

fig = px.imshow(
    conf_svm,
    text_auto=True,
    aspect='auto',
    template='seaborn',
    x=classes,
    y=classes,
    labels= dict(x='Predição', y='Real', color='Contagem'),
    color_continuous_scale= color_scale,
    title='Matriz de confusão - SVM'

)
fig.show()

Relatório de classificação do SVM
              precision    recall  f1-score   support

           0       1.00      0.99      1.00     85295
           1       0.17      0.88      0.28       148

    accuracy                           0.99     85443
   macro avg       0.58      0.94      0.64     85443
weighted avg       1.00      0.99      0.99     85443



In [22]:
# Teste do modelo de Random Forest
y_pred_rf = best_rf.predict(x_test)

# Avaliação do modelo com o teste
report_rf = classification_report(y_test, y_pred_rf)
print('Relatório de classificação do Random Forest')
print(report_rf)

# Matriz de confusão
conf_rf = confusion_matrix(y_test, y_pred_rf)
classes = ['Legítimo', 'Fraude']

fig = px.imshow(
    conf_rf,
    text_auto=True,
    aspect='auto',
    template='seaborn',
    x=classes,
    y=classes,
    labels= dict(x='Predição', y='Real', color='Contagem'),
    color_continuous_scale= color_scale,
    title='Matriz de confusão - Random Forest'

)
fig.show()

Relatório de classificação do Random Forest
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     85295
           1       0.83      0.82      0.83       148

    accuracy                           1.00     85443
   macro avg       0.92      0.91      0.91     85443
weighted avg       1.00      1.00      1.00     85443



In [23]:
# Teste do modelo de XGBoost
y_pred_xgb = best_xgb.predict(x_test)

# Avaliação do modelo com o teste
report_xgb = classification_report(y_test, y_pred_xgb)
print('Relatório de classificação XGBoost')
print(report_xgb)

# Matriz de confusão
conf_xgb = confusion_matrix(y_test, y_pred_xgb)
classes = ['Legítimo', 'Fraude']

fig = px.imshow(
    conf_xgb,
    text_auto=True,
    aspect='auto',
    template='seaborn',
    x=classes,
    y=classes,
    labels= dict(x='Predição', y='Real', color='Contagem'),
    color_continuous_scale= color_scale,
    title='Matriz de confusão - XGBoost'

)
fig.show()

Relatório de classificação XGBoost
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     85295
           1       0.73      0.82      0.77       148

    accuracy                           1.00     85443
   macro avg       0.87      0.91      0.89     85443
weighted avg       1.00      1.00      1.00     85443



Os modelos tiveram resultados satisfatórios, porém diferentes. A escolha do modelo deve se basear na preferência do banco, pois todos apresentaram pontos fortes e fracos.
* Regressão logística - errou somente 18 dados classificados como fraude (12%), mas classificou erroneamente 796 transções legítimas como fraudes (~1%);
* SVM - igual à Regressão logística, porém com menos erros nas transações legítimas: 654(~1%);
* Random Forest - 9 erros a mais nas classificações de fraudes (+6%), mas somente 24 classificações errôneas de transações legítimas (0.02%);
* XGBoost - igual ao Random Forest, mas com acréscimo leve nas classificações erradas de transações legítimas +20(+ ~0.03%).

In [31]:
# Curva AUC-ROC de previsão

models = [best_lr, best_svm, best_rf, best_xgb]
model_names = ["Regressão Logística", "SVM", "RandomForest", "XGBoost"]


X = x_test
Y = y_test


def get_score(model, X):

    # 1) predict_proba
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)

        if proba.ndim == 2 and proba.shape[1] >= 2:
            return proba[:, 1]

        return proba.ravel()

    # 2) decision_function
    if hasattr(model, "decision_function"):
        dfs = model.decision_function(X)

        if getattr(dfs, "ndim", 1) == 1:
            return dfs
        return dfs[:, -1]

    # 3) tentar calibrar (pode ser mais lento)
    try:
        calibrated = CalibratedClassifierCV(model, cv=5)
        calibrated.fit(x_train_resampled, y_train_resampled)
        return calibrated.predict_proba(X)[:, 1]
    except Exception as e:
        raise RuntimeError(f"Não foi possível obter scores do modelo: {e}")


fig = go.Figure()
fig.add_shape(type='line', line=dict(dash='dash', color='gray'), x0=0, x1=1, y0=0, y1=1)


for model, name in zip(models, model_names):
    # obtém vetor de scores contínuos
    scores = get_score(model, X)

    # valida shapes
    if scores.ndim != 1:
        scores = np.asarray(scores).ravel()

    fpr, tpr, _ = roc_curve(Y, scores)
    auc_score = roc_auc_score(Y, scores)

    label = f"{name} (AUC={auc_score:.3f})"
    fig.add_trace(go.Scatter(x=fpr, y=tpr, name=label, mode='lines'))


fig.update_layout(
    title="Comparação ROC AUC - Modelos",
    xaxis=dict(title="Falso positivo", constrain='domain'),
    yaxis=dict(title="Verdadeiro Positivo", scaleanchor='x', scaleratio=1),
    width=1280, height=720,
    legend=dict(title="Modelos")
)
fig.show()


In [36]:
# Quantia salva por operação geral
save_op = (df['Amount'].mean()) * 0.98
print("Quantidade salva, em média, por operação geral:", save_op.round(2),"/",df['Amount'].mean().round(2))

# Quantia salva por operação fraudulenta
frau_op = (fraude['Amount'].mean()) * 0.82
print("Quantidade salva, em média, por operação Fradudulenta:", frau_op.round(2),"/",fraude['Amount'].mean().round(2))

# Quantia salva pelo banco em termos gerais
save_gr = (df['Amount'].sum())*  0.98
print("Quantidade monetária salva em termos gerais: ", save_gr.round(2))

# Quantia salva pelo banco com detecção de fraudes
frau_save = (fraude['Amount'].sum())* 0.82
print("Quantidade monetária salva com detecção de fraudes: ", frau_save.round(2))

Quantidade salva, em média, por operação geral: 86.58 / 88.35
Quantidade salva, em média, por operação Fradudulenta: 100.21 / 122.21
Quantidade monetária salva em termos gerais:  24659338.21
Quantidade monetária salva com detecção de fraudes:  49304.94


# Resultado
Levando em consideração a performance geral dos modelos, capacidade de identificar classes corretamente e capacidade de identificar operações fraudulentas, o modelo de XGBoost deve ajudar o banco a salvar uma cerca de 86.00 por operação e 100.00 por operação fraudulenta, em média. Isso representa um bom ganho para o banco nas suas operações diárias.